# 160-channel + 1000× decimation + flattop → single H5

**No PCA** — keep all 160 channels (20×8 ECEi sensor array).

**Pipeline:** ecei_mc decimated H5 (already 10× → 100 kHz) → crop to flattop → additional 100× decimation (total 1000× → 1 kHz) → tile into subsequences → save to H5.

**Subsequence shape:** `(160, 781)` — 781,250 raw samples / 1000 = 781 at 1 kHz.

**Output:** `/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc/160ch_1000x_flattop/all_data.h5`

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────
BASE = Path("/home/idies/workspace/Storage/yhuang2/persistent/ecei_mc")
CLEAR_DECIMATED = BASE / "clear_decimated"
DISRUPT_DECIMATED = BASE / "disrupt_decimated"

DISRUPT_SHOT_LIST = Path("disruptcnn/shots/d3d_disrupt_ecei.final.txt")
CLEAR_SHOT_LIST = Path("disruptcnn/shots/d3d_clear_ecei.final.txt")

OUT_DIR = BASE / "160ch_1000x_flattop"
OUT_H5 = OUT_DIR / "all_data.h5"

# Decimation: ecei_mc is 10× decimated; we do another 100× → total 1000×
DATA_STEP = 10            # existing (raw 1 MHz → 100 kHz in H5)
EXTRA_DECIMATE = 100      # additional (100 kHz → 1 kHz)
TOTAL_DECIMATE = DATA_STEP * EXTRA_DECIMATE  # 1000

# Subsequence params
NSUB_RAW = 781_250
STRIDE_RAW = 481_090
T_SUB = NSUB_RAW // TOTAL_DECIMATE     # 781
STRIDE = STRIDE_RAW // TOTAL_DECIMATE  # 481
CHANNELS = 20 * 8  # 160

# Labels
TWARN_MS = 300.0
TWARN_RAW = int(TWARN_MS * 1000)  # 300,000 raw samples

# Splits
TRAIN_FRAC, VAL_FRAC = 0.8, 0.1
RANDOM_SEED = 42

# Shot list columns
COL_SHOT, COL_TSTART, COL_TLAST, COL_DT = 0, 2, 3, 4
COL_T_FLAT_START, COL_T_FLAT_LAST, COL_TDISRUPT = 6, 7, 8

print(f"T_SUB = {T_SUB}, STRIDE = {STRIDE}")
print(f"Total decimate = {TOTAL_DECIMATE}× (1 MHz → 1 kHz)")
print(f"Channels = {CHANNELS} (no PCA)")
print(f"Output: {OUT_H5}")

## 1. Parse shot lists, filter valid flattop, stratified split

In [ ]:
def parse_shot_list(path: Path) -> pd.DataFrame:
    data = np.loadtxt(path, skiprows=1)
    if data.ndim == 1:
        data = data[np.newaxis, :]
    df = pd.DataFrame({
        "shot": data[:, COL_SHOT].astype(int),
        "tstart_ms": data[:, COL_TSTART],
        "tlast_ms": data[:, COL_TLAST],
        "dt_ms": data[:, COL_DT],
        "t_flat_start_ms": data[:, COL_T_FLAT_START],
        "t_flat_last_ms": data[:, COL_T_FLAT_LAST],
        "tdisrupt_ms": data[:, COL_TDISRUPT],
    })
    df["t_flat_stop_ms"] = df["t_flat_start_ms"] + df["t_flat_last_ms"]
    df["flat_start_dec"] = (df["t_flat_start_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP
    df["flat_stop_dec"] = (df["t_flat_stop_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP
    df["tlast_dec"] = (df["tlast_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP
    df["tdisrupt_dec"] = np.where(
        df["tdisrupt_ms"] > 0,
        (df["tdisrupt_ms"] - df["tstart_ms"]) / df["dt_ms"] / DATA_STEP, -1.0)
    return df

df_disrupt = parse_shot_list(DISRUPT_SHOT_LIST)
df_clear = parse_shot_list(CLEAR_SHOT_LIST)
disrupt_info = df_disrupt.set_index("shot")
clear_info = df_clear.set_index("shot")

def list_h5_shots(root):
    if not root.exists(): return []
    return sorted(int(p.stem) for p in root.glob("*.h5") if p.stem.isdigit())

def filter_valid_flattop(shots, info_df):
    valid = []
    for s in shots:
        if s not in info_df.index: continue
        row = info_df.loc[s]
        if pd.isna(row["flat_start_dec"]) or pd.isna(row["flat_stop_dec"]): continue
        if row["flat_stop_dec"] <= row["flat_start_dec"]: continue
        valid.append(s)
    return valid

def stratified_split(shot_ids, train_frac, val_frac, seed):
    n = len(shot_ids)
    if n == 0: return [], [], []
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    n_train, n_val = int(n * train_frac), int(n * val_frac)
    return ([shot_ids[i] for i in perm[:n_train]],
            [shot_ids[i] for i in perm[n_train:n_train+n_val]],
            [shot_ids[i] for i in perm[n_train+n_val:]])

disrupt_valid = filter_valid_flattop(list_h5_shots(DISRUPT_DECIMATED), disrupt_info)
clear_valid = filter_valid_flattop(list_h5_shots(CLEAR_DECIMATED), clear_info)
disrupt_train, disrupt_val, disrupt_test = stratified_split(disrupt_valid, TRAIN_FRAC, VAL_FRAC, RANDOM_SEED)
clear_train, clear_val, clear_test = stratified_split(clear_valid, TRAIN_FRAC, VAL_FRAC, RANDOM_SEED)

print(f"Disrupt: {len(disrupt_valid)} valid → train {len(disrupt_train)}, val {len(disrupt_val)}, test {len(disrupt_test)}")
print(f"Clear:   {len(clear_valid)} valid → train {len(clear_train)}, val {len(clear_val)}, test {len(clear_test)}")

## 2. Load helper: crop to flattop, decimate 100× (total 1000× from raw), keep 160 channels

In [ ]:
def get_flattop_bounds(shot, is_clear):
    info = disrupt_info if not is_clear else clear_info
    row = info.loc[shot]
    flat_start = int(row["flat_start_dec"])
    flat_stop = int(row["flat_stop_dec"])
    tlast = int(row["tlast_dec"])
    tdis = int(row["tdisrupt_dec"])
    if is_clear or tdis < 0:
        return flat_start, min(tlast, flat_stop), -1
    else:
        return flat_start, max(tdis, min(tlast, flat_stop)), tdis


def load_flattop_1000x(root, shot, is_clear):
    """Load shot, crop to flattop, decimate 100× → (160, T_1000x).

    Returns:
        X: (160, T_1000x) float32
        tdisrupt_1000x: disruption index in 1000× space relative to flattop start; -1 if clear
    """
    flat_start, flat_end, tdis_dec = get_flattop_bounds(shot, is_clear)
    with h5py.File(root / f"{shot}.h5", "r") as f:
        T_file = f["LFS"].shape[-1]
        flat_start = max(0, min(flat_start, T_file))
        flat_end = max(flat_start, min(flat_end, T_file))
        data = np.asarray(f["LFS"][..., flat_start:flat_end], dtype=np.float32)  # (20, 8, T_flat)

    # Decimate by EXTRA_DECIMATE (100×)
    data = data[..., ::EXTRA_DECIMATE]  # (20, 8, T_1000x)
    T_out = data.shape[-1]
    X = data.reshape(CHANNELS, T_out)  # (160, T_1000x)

    if tdis_dec >= 0:
        tdisrupt_1000x = max(0, (tdis_dec - flat_start) // EXTRA_DECIMATE)
    else:
        tdisrupt_1000x = -1
    return X, tdisrupt_1000x


# Quick test
if disrupt_valid:
    _X, _td = load_flattop_1000x(DISRUPT_DECIMATED, disrupt_valid[0], False)
    print(f"Test disrupt shot {disrupt_valid[0]}: X.shape={_X.shape}, tdisrupt_1000x={_td}")
if clear_valid:
    _X, _td = load_flattop_1000x(CLEAR_DECIMATED, clear_valid[0], True)
    print(f"Test clear shot {clear_valid[0]}: X.shape={_X.shape}, tdisrupt_1000x={_td}")

## 3. Build subsequences and compute labels/weights

Tile each shot into subsequences of length `T_SUB=781` with stride `STRIDE=481`.
Per-timestep target and weight using Twarn=300ms (= 300 samples at 1 kHz).

In [ ]:
def compute_target_weight(T, tdisrupt, twarn):
    target = np.zeros(T, dtype=np.float32)
    weight = np.ones(T, dtype=np.float32)
    if tdisrupt < 0:
        return target, weight
    warn_start = max(0, tdisrupt - twarn)
    warn_end = min(T, tdisrupt)
    if warn_start < T and warn_end > warn_start:
        target[warn_start:warn_end] = 1.0
    if tdisrupt < T:
        weight[tdisrupt:] = 0.0
    return target, weight


def process_shot(root, shot, is_clear):
    try:
        X, tdis = load_flattop_1000x(root, shot, is_clear)
    except (OSError, IOError, KeyError):
        return []
    T_total = X.shape[1]
    if T_total < T_SUB:
        return []

    twarn = TWARN_RAW // TOTAL_DECIMATE  # 300,000 / 1000 = 300

    results = []
    pos = 0
    while pos + T_SUB <= T_total:
        chunk = X[:, pos:pos + T_SUB]  # (160, T_SUB)
        td_local = tdis - pos if tdis >= 0 else -1
        target, weight = compute_target_weight(T_SUB, td_local, twarn)
        has_disrupt = 1 if target.sum() > 0 else 0
        results.append((chunk, target, weight, has_disrupt, shot))
        pos += STRIDE

    last_start = T_total - T_SUB
    if last_start > (pos - STRIDE):
        chunk = X[:, last_start:T_total]
        td_local = tdis - last_start if tdis >= 0 else -1
        target, weight = compute_target_weight(T_SUB, td_local, twarn)
        has_disrupt = 1 if target.sum() > 0 else 0
        results.append((chunk, target, weight, has_disrupt, shot))

    return results


split_map = {
    "train": [(DISRUPT_DECIMATED, s, False) for s in disrupt_train] +
             [(CLEAR_DECIMATED, s, True) for s in clear_train],
    "val":   [(DISRUPT_DECIMATED, s, False) for s in disrupt_val] +
             [(CLEAR_DECIMATED, s, True) for s in clear_val],
    "test":  [(DISRUPT_DECIMATED, s, False) for s in disrupt_test] +
             [(CLEAR_DECIMATED, s, True) for s in clear_test],
}

all_data = {}
for split_name, shot_tuples in split_map.items():
    X_list, tgt_list, wgt_list, lbl_list, sid_list = [], [], [], [], []
    for root, shot, is_clear in tqdm(shot_tuples, desc=split_name):
        for chunk, target, weight, label, sid in process_shot(root, shot, is_clear):
            X_list.append(chunk)
            tgt_list.append(target)
            wgt_list.append(weight)
            lbl_list.append(label)
            sid_list.append(sid)
    if X_list:
        all_data[split_name] = {
            "X": np.stack(X_list),                          # (N, 160, T_SUB)
            "target": np.stack(tgt_list),                   # (N, T_SUB)
            "weight": np.stack(wgt_list),                   # (N, T_SUB)
            "labels": np.array(lbl_list, dtype=np.int64),   # (N,)
            "shot_ids": np.array(sid_list, dtype=np.int64), # (N,)
        }
        n_pos = sum(lbl_list)
        print(f"  [{split_name}] {len(X_list)} subseqs ({n_pos} disruptive, {len(X_list)-n_pos} clear)")
    else:
        print(f"  [{split_name}] 0 subsequences")

## 4. Write to single H5

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Class weights
if "train" in all_data:
    t = all_data["train"]["target"]
    w = all_data["train"]["weight"]
    total_pos = float((t * w).sum())
    total_neg = float(((1 - t) * w).sum())
    total = total_pos + total_neg
    pos_weight = float(0.5 * total / total_pos) if total_pos > 0 else 1.0
    neg_weight = float(0.5 * total / total_neg) if total_neg > 0 else 1.0
    print(f"Class weights: pos={pos_weight:.4f}, neg={neg_weight:.4f}")
else:
    pos_weight = neg_weight = 1.0

with h5py.File(OUT_H5, "w") as f:
    for split_name, data in all_data.items():
        g = f.create_group(split_name)
        g.create_dataset("X", data=data["X"], dtype=np.float32)
        g.create_dataset("target", data=data["target"], dtype=np.float32)
        g.create_dataset("weight", data=data["weight"], dtype=np.float32)
        g.create_dataset("labels", data=data["labels"], dtype=np.int64)
        g.create_dataset("shot_ids", data=data["shot_ids"], dtype=np.int64)

    f.attrs["nsub_raw"] = NSUB_RAW
    f.attrs["stride_raw"] = STRIDE_RAW
    f.attrs["data_step"] = DATA_STEP
    f.attrs["extra_decimate"] = EXTRA_DECIMATE
    f.attrs["total_decimate"] = TOTAL_DECIMATE
    f.attrs["t_sub"] = T_SUB
    f.attrs["stride"] = STRIDE
    f.attrs["channels"] = CHANNELS
    f.attrs["twarn_ms"] = TWARN_MS
    f.attrs["pos_weight"] = pos_weight
    f.attrs["neg_weight"] = neg_weight
    f.attrs["random_seed"] = RANDOM_SEED
    for split_name in all_data:
        f.attrs[f"n_{split_name}"] = len(all_data[split_name]["labels"])

import os
size_mb = os.path.getsize(OUT_H5) / (1024 ** 2)
print(f"\nSaved to {OUT_H5}")
print(f"File size: {size_mb:.1f} MB")

## 5. Verify

In [ ]:
with h5py.File(OUT_H5, "r") as f:
    print("=== H5 structure ===")
    for key in f.keys():
        if isinstance(f[key], h5py.Group):
            print(f"  /{key}/")
            for dset in f[key]:
                print(f"    {dset}: {f[key][dset].shape} {f[key][dset].dtype}")

    print("\n=== Metadata ===")
    for attr in f.attrs:
        print(f"  {attr}: {f.attrs[attr]}")

    print("\n=== Split stats ===")
    for split in ("train", "val", "test"):
        if split not in f: continue
        X = f[split]["X"]
        labels = np.asarray(f[split]["labels"])
        n = len(labels)
        n_pos = int(labels.sum())
        print(f"  {split}: {n} subseqs, X.shape={X.shape}, "
              f"disruptive={n_pos}, clear={n - n_pos}")

    if "train" in f:
        X0 = np.asarray(f["train/X"][0])
        t0 = np.asarray(f["train/target"][0])
        w0 = np.asarray(f["train/weight"][0])
        print(f"\n=== First train subseq ===")
        print(f"  X: shape={X0.shape}, min={X0.min():.3f}, max={X0.max():.3f}")
        print(f"  target: {t0.sum():.0f} positive / {len(t0)}")
        print(f"  weight: nonzero={int((w0 > 0).sum())} / {len(w0)}")